# Рынок жилой недвижимости Казани: подготовка данных

В этом ноутбуке из четырех квартальных файлов Росреестра по всей России отбираются записи о жилых помещениях в Казани. Полученный локальный набор данных используется в дальнейшем анализе. Исходные файлы обрабатываются по частям, чтобы ограничить использование оперативной памяти.

In [21]:
from pathlib import Path

import pandas as pd


# Пути корректны при запуске Jupyter из корня проекта или из папки notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_FILE = DATA_DIR / "kazan_residential_transactions_2025_raw.csv"
KAZAN_OKATO_PREFIX = "92401"
PREMISES_CODE = "002001003000"
RESIDENTIAL_PURPOSE_CODE = "206002000000"
CHUNK_SIZE = 100_000

## Исходные файлы

Для анализа необходим один исходный файл по всей России за каждый квартал 2025 года.

In [22]:
csv_files = sorted(DATA_DIR.glob("dataset_СДЕЛКИ_*.csv"))

if len(csv_files) != 4:
    raise ValueError(
        f"Ожидалось 4 квартальных файла, найдено {len(csv_files)} в {DATA_DIR}"
    )

for file_path in csv_files:
    print(file_path.name)

dataset_СДЕЛКИ_r-r_01-92_y_2025_q_1.csv
dataset_СДЕЛКИ_r-r_01-92_y_2025_q_2.csv
dataset_СДЕЛКИ_r-r_01-92_y_2025_q_3.csv
dataset_СДЕЛКИ_r-r_01-92_y_2025_q_4.csv


## Отбор жилых помещений в Казани

Казань определяется по префиксу ОКАТО `92401`. Затем остаются объекты с типом «Помещение» (`002001003000`) и назначением «Жилое помещение» (`206002000000`). Это наиболее близкое доступное приближение к рынку квартир: в источнике нет отдельного признака, позволяющего надежно отличить квартиры от комнат. При полном повторном запуске ноутбука результирующий файл создается заново.

In [23]:
summary_rows = []
write_header = True

for file_path in csv_files:
    raw_row_count = 0
    residential_row_count = 0
    quarter = f"Q{file_path.stem[-1]}"

    for chunk in pd.read_csv(
        file_path,
        sep="~",
        encoding="utf-8",
        dtype="string",
        chunksize=CHUNK_SIZE,
    ):
        raw_row_count += len(chunk)

        is_kazan = chunk["okato"].str.startswith(
            KAZAN_OKATO_PREFIX,
            na=False,
        )
        is_residential_premises = (
            (chunk["realestate_type_code"] == PREMISES_CODE)
            & (chunk["purpose_code"] == RESIDENTIAL_PURPOSE_CODE)
        )
        target_rows = is_kazan & is_residential_premises
        residential_chunk = chunk.loc[target_rows].copy()
        residential_row_count += len(residential_chunk)

        if len(residential_chunk) > 0:
            residential_chunk["quarter"] = quarter

            residential_chunk.to_csv(
                OUTPUT_FILE,
                mode="w" if write_header else "a",
                header=write_header,
                index=False,
                encoding="utf-8",
            )
            write_header = False

    summary_rows.append(
        {
            "квартал": quarter,
            "строк_в_источнике": raw_row_count,
            "строк_жилых_помещений": residential_row_count,
        }
    )

load_summary = pd.DataFrame(summary_rows)
load_summary

,quarter,raw_rows,residential_rows
0,Q1,542271,1505
1,Q2,638474,1102
2,Q3,713281,1355
3,Q4,777632,1622


## Проверка сохраненного набора данных

Проверяются размер набора данных, покрытие кварталов, географические границы, тип объекта и назначение помещения.

In [24]:
residential_raw = pd.read_csv(
    OUTPUT_FILE,
    dtype="string",
    encoding="utf-8",
)

non_kazan_rows = (
    ~residential_raw["okato"].str.startswith(KAZAN_OKATO_PREFIX, na=False)
).sum()
non_premises_rows = (
    residential_raw["realestate_type_code"] != PREMISES_CODE
).sum()
non_residential_rows = (
    residential_raw["purpose_code"] != RESIDENTIAL_PURPOSE_CODE
).sum()

print(f"Сохраненный файл: {OUTPUT_FILE}")
print(f"Размер набора данных: {residential_raw.shape}")
print(f"Строк за пределами Казани: {non_kazan_rows}")
print(f"Строк с типом объекта не «Помещение»: {non_premises_rows}")
print(f"Строк с нежилым назначением: {non_residential_rows}")
print()
print("Количество строк по кварталам:")
print(residential_raw["quarter"].value_counts().sort_index())

residential_raw.head()

Saved file: c:\Users\Artem\Documents\Kazan_Housing\Kazan-housing\data\kazan_residential_transactions_2025_raw.csv
Dataset shape: (5584, 18)
Non-Kazan rows: 0
Non-premises rows: 0
Non-residential rows: 0

Rows by quarter:
quarter
Q1    1505
Q2    1102
Q3    1355
Q4    1622
Name: count, dtype: Int64


,number,okato,region_code,district,city,quarter_cad_number,street,realestate_type_code,wall_material_code,year_build,floor,purpose_code,area,period_start_date,deal_price,currency,doc_type,quarter
0,1,92401380000,16,<NA>,Казань,16:50:160205,Хусаина Мавлютова,002001003000,061001001001,1986,9,206002000000,56.9,2025-01-01,8900000.0,рубль,ДКП,Q1
1,3,92401367000,16,<NA>,Казань,16:50:011102,Старая,002001003000,<NA>,<NA>,<NA>,206002000000,7991.3099999999995,2025-01-01,54923916,рубль,ДДУ,Q1
2,1,92401367000,16,<NA>,Казань,16:50:011705,Николая Столбова,002001003000,061001003000,2020,2,206002000000,62.1,2025-01-01,26400000,рубль,ДКП,Q1
3,1,92401385000,16,<NA>,Казань,16:50:060510,Комарова,002001003000,061001007001,1969,5,206002000000,59.4,2025-01-01,9000000.0,рубль,ДКП,Q1
4,1,92401363000,16,<NA>,Казань,16:50:220526,Молодежная,002001003000,061001001001,1982,5,206002000000,12.9,2025-01-01,2200000.0,рубль,ДКП,Q1
